<a href="https://colab.research.google.com/github/ejdansu/Python/blob/main/Artificial_Intelligence_for_Equatorial_Space_Weather_From_Data_to_Operational_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# International Colloquium on Equatorial and Low-Latitude Ionosphere (ICELLI) 2026
Lead City University, Ibadan, Nigeria

Monday, 31 August — Friday, 4 September, 2026.

---
# Artificial Intelligence for Equatorial Space Weather: From Data to Operational Prediction

_A case study in AI-driven prediction of Total Electron Content (TEC)_

---

### Instructor

**Dr. Emmanuel J. Dansu**,

Tohoku University, Sendai, Japan

ejdansu@tohoku.ac.jp; ejdansu@gmail.com

---

## Duration

45 Minutes

---

## Objectives

In this workshop we will learn how to

- understand the equatorial ionosphere
- generate a realistic space weather dataset
- build Machine Learning models
- compare AI algorithms
- interpret predictions
- forecast Total Electron Content (TEC)

using Python.

# Motivation

The equatorial ionosphere is one of the most dynamic regions of near-Earth space.

Its variability influences

- GPS positioning
- Satellite communications
- HF radio propagation
- Space weather services

Modern space weather missions continuously produce enormous volumes of observations.

Artificial Intelligence provides powerful tools for learning complex nonlinear relationships within these observations.

Rather than replacing physics,

Artificial Intelligence complements scientific understanding.

# Scientific Question

Can Machine Learning predict

## Total Electron Content (TEC)

using measurements of

- Solar Activity
- Geomagnetic Activity
- Solar Wind
- Local Time
- Seasonal Variation

If successful,

the same workflow can later be applied to

- foF2
- hmF2
- Plasma Bubbles
- Spread-F
- Scintillation
- GNSS Positioning Error

# Machine Learning Workflow

Every Machine Learning project follows the same workflow.

```

Scientific Question

↓

Collect Data

↓

Explore Data

↓

Prepare Features

↓

Train AI

↓

Evaluate AI

↓

Interpret Results

↓

Operational Prediction

```

We will complete this entire workflow.

In [ ]:
# ==========================================================
# Import Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Reproducibility
np.random.seed(42)

plt.style.use("ggplot")

print("Libraries loaded successfully.")

# Part 1 — Creating a Synthetic Equatorial Space Weather Dataset

For this live tutorial we use a **synthetic but physically inspired** dataset.

Why?

- No internet connection is required.
- Everyone obtains identical results.
- The notebook is fully reproducible.
- The relationships mimic known equatorial ionospheric behaviour.

The dataset includes

- Solar Flux (F10.7)
- Dst Index
- Kp Index
- IMF Bz
- Solar Wind Speed
- Solar Wind Density
- Local Time
- Day of Year
- Total Electron Content (TEC)

In [ ]:
# ==========================================================
# Generate Time Variables
# ==========================================================

N = 8760          # One year of hourly observations

Hour = np.arange(N) % 24

DayOfYear = (np.arange(N) // 24) % 365 + 1

print("Number of observations:", N)

In [ ]:
# ==========================================================
# Solar Flux (F10.7)
# ==========================================================

F107 = (
    150
    + 30 * np.sin(2 * np.pi * DayOfYear / 365)
    + np.random.normal(0, 5, N)
)

F107 = np.clip(F107, 80, 250)

In [ ]:
# ==========================================================
# Dst Index
# ==========================================================

Dst = np.random.normal(-10, 15, N)

storm_index = np.random.choice(
    N,
    120,
    replace=False
)

Dst[storm_index] -= np.random.uniform(
    40,
    150,
    len(storm_index)
)

In [ ]:
# ==========================================================
# Kp Index
# ==========================================================

Kp = (
    2
    + 0.02 * (-Dst)
    + np.random.normal(0, 0.5, N)
)

Kp = np.clip(Kp, 0, 9)

In [ ]:
# ==========================================================
# IMF Bz
# ==========================================================

Bz = np.random.normal(
    0,
    4,
    N
)

In [ ]:
# ==========================================================
# Solar Wind Speed
# ==========================================================

Vsw = (
    420
    + 0.8 * (-Dst)
    + np.random.normal(
        0,
        30,
        N
    )
)

Vsw = np.clip(
    Vsw,
    300,
    800
)

In [ ]:
# ==========================================================
# Proton Density
# ==========================================================

Density = np.random.normal(
    6,
    2,
    N
)

Density = np.clip(
    Density,
    1,
    20
)

# Creating Synthetic TEC

We now construct a synthetic TEC variable that reflects several well-known physical influences:

- **Solar activity (F10.7):** Higher solar flux generally increases ionization and TEC.
- **Local time:** TEC follows a strong diurnal cycle, peaking during the daytime.
- **Season:** Seasonal changes influence ionospheric density.
- **Geomagnetic storms (Dst):** Disturbances can either enhance or deplete TEC.
- **Southward IMF Bz:** Negative Bz strengthens solar wind–magnetosphere coupling.

Finally, we add a small amount of random noise to represent measurement uncertainty.

In [ ]:
# ==========================================================
# Diurnal Variation
# ==========================================================

Diurnal = 15 * np.sin(
    2 * np.pi * (Hour - 6) / 24
)

# Seasonal Variation
Seasonal = 5 * np.cos(
    2 * np.pi * DayOfYear / 365
)

# Storm-Time Effect
Storm = 0.05 * Dst

# Southward Bz Effect
BzEffect = -0.4 * np.minimum(Bz, 0)

# Measurement Noise
Noise = np.random.normal(0, 2, N)

# Synthetic TEC
TEC = (
    20
    + 0.12 * F107
    + Diurnal
    + Seasonal
    + Storm
    + BzEffect
    + Noise
)

TEC = np.clip(TEC, 5, 90)

In [ ]:
# ==========================================================
# Build the DataFrame
# ==========================================================

data = pd.DataFrame({
    "Hour": Hour,
    "DayOfYear": DayOfYear,
    "F107": F107,
    "Dst": Dst,
    "Kp": Kp,
    "Bz": Bz,
    "Vsw": Vsw,
    "Density": Density,
    "TEC": TEC
})

print(data.shape)
data.head()

# Part 2 — Exploratory Data Analysis

Machine Learning should never be the first step in a scientific investigation.

Before building predictive models, we must first understand the data.

Exploratory Data Analysis (EDA) allows us to:

- inspect the observations,
- identify unrealistic values,
- understand the distribution of each variable,
- explore relationships between variables,
- and develop physical intuition.

> **Good Machine Learning begins with good data.**

In [ ]:
# ==========================================================
# Display the First Five and Last Five Observations
# ==========================================================

data

Each row represents one hourly observation of the equatorial ionosphere.

Notice that:

- Each predictor corresponds to a measurable physical quantity.
- The target variable is **TEC**.
- Our goal is to learn the relationship between the predictors and TEC.

In [ ]:
# ==========================================================
# Dataset Information
# ==========================================================

data.info()

In [ ]:
# ==========================================================
# Summary Statistics
# ==========================================================

data.describe().round(2)

## Discussion

The summary statistics provide useful information such as

- minimum values,
- maximum values,
- mean,
- standard deviation,
- quartiles.

Ask yourself:

- Do the values appear physically reasonable?
- Are there obvious outliers?
- Do any variables require scaling?

In [ ]:
# ==========================================================
# Missing Values
# ==========================================================

data.isnull().sum()

Our synthetic dataset was generated without missing values.

In real-world space weather datasets, handling missing observations is often one of the most time-consuming parts of the analysis.

# Distribution of Variables

Understanding the distribution of each variable helps us answer questions such as

- Is the variable approximately normal?
- Is it strongly skewed?
- Does it contain extreme values?

These characteristics influence model selection and performance.

In [ ]:
columns = [
    "TEC",
    "F107",
    "Dst",
    "Kp",
    "Bz",
    "Vsw",
    "Density"
]

data[columns].hist(
    figsize=(12,8),
    bins=25
)

plt.tight_layout()
plt.show()

## Scientific Interpretation

Several important observations can already be made.

---

### TEC

Shows strong variability resulting from

- local time,
- solar activity,
- geomagnetic activity.

---

### F10.7

Varies gradually with season and represents changing solar activity.

---

### Dst

Contains occasional large negative values corresponding to geomagnetic storms.

---

### Kp

Mostly represents quiet-to-moderate geomagnetic conditions.

---

### Solar Wind Speed

Shows increased values during disturbed periods.

---


# Visualising TEC

The variable we wish to predict is **Total Electron Content (TEC)**.

Before building a predictive model, we should examine how TEC changes over time.

In [ ]:
plt.figure(figsize=(14,4))

plt.plot(
    data["TEC"],
    linewidth=0.8,
)

plt.xlabel("Observation")

plt.ylabel("TEC (TECU)")

plt.title("Synthetic Equatorial Total Electron Content")

plt.show()

## Interpretation

Several patterns are evident.

- A clear diurnal cycle.
- Seasonal modulation.
- Increased variability during disturbed conditions.

These characteristics are consistent with our understanding of the equatorial ionosphere.

The next question becomes:

**Can Artificial Intelligence learn these patterns automatically?**

# Correlation Analysis

Before applying Machine Learning, we examine linear relationships among the variables.

Correlation is **not** the same as causation, and weak linear correlation does **not** imply that a variable is unimportant.

Many ionospheric processes are inherently nonlinear.

In [ ]:
corr = data.corr(numeric_only=True)

corr.round(2)

In [ ]:
plt.figure(figsize=(8,6))

im = plt.imshow(
    corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.colorbar(im)

plt.xticks(
    range(len(corr.columns)),
    corr.columns,
    rotation=90
)

plt.yticks(
    range(len(corr.columns)),
    corr.columns
)

plt.title("Correlation Matrix")

plt.tight_layout()

plt.show()

The correlation matrix provides a useful overview, but it cannot capture complex interactions between variables.

This is precisely where Machine Learning becomes valuable.

# Pairwise Relationships

Scatter plots help us visualise how individual predictors relate to TEC.

Although the relationships may appear noisy, Machine Learning algorithms can often learn complex nonlinear patterns that are not obvious to the human eye.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10,8))

pairs = [
    ("F107", "TEC"),
    ("Dst", "TEC"),
    ("Kp", "TEC"),
    ("Bz", "TEC"),
]

for ax, (x, y) in zip(axes.ravel(), pairs):
    ax.scatter(
        data[x],
        data[y],
        alpha=0.4,
        s=8
    )
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{y} vs {x}")

plt.tight_layout()
plt.show()

# Part 3 — Building Machine Learning Models

After exploring the data, we are now ready to build predictive models.

Our objective is to predict

# Total Electron Content (TEC)

using

- Local Time
- Season
- Solar Activity
- Geomagnetic Activity
- Solar Wind Conditions

We begin with a simple baseline model before introducing more advanced algorithms.

# Feature Matrix and Target Variable

Machine Learning distinguishes between

## Features (Inputs)

The variables used to make predictions.

For this study

- Hour
- DayOfYear
- F107
- Dst
- Kp
- IMF Bz
- Solar Wind Speed
- Proton Density

---

## Target

The variable we wish to predict

**TEC**

In [ ]:
# ==========================================================
# Define Features and Target
# ==========================================================

features = [
    "Hour",
    "DayOfYear",
    "F107",
    "Dst",
    "Kp",
    "Bz",
    "Vsw",
    "Density"
]

X = data[features]

y = data["TEC"]

print("Feature Matrix:", X.shape)
print("Target Vector :", y.shape)

X.head()

## Why These Features?

Each predictor represents a physical process.

| Feature | Physical Meaning |
|----------|------------------|
| Hour | Diurnal ionization cycle |
| DayOfYear | Seasonal effects |
| F107 | Solar EUV activity |
| Dst | Storm intensity |
| Kp | Geomagnetic activity |
| Bz | Solar wind coupling |
| Vsw | Solar wind energy |
| Density | Solar wind plasma |

Rather than choosing variables arbitrarily,

we selected them because ionospheric physics suggests that they influence TEC.

# Training and Testing Data

To evaluate whether the model truly learns,

we divide the dataset into

- **80% Training Data**
- **20% Testing Data**

The testing data are never shown during training.

This prevents overly optimistic estimates of model performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42

)

print("Training samples :", len(X_train))

print("Testing samples  :", len(X_test))

# Baseline Model — Linear Regression

A good scientific workflow begins with the simplest reasonable model.

Linear Regression assumes that TEC changes approximately linearly with each predictor.

Although we know that the ionosphere behaves nonlinearly,

this baseline allows us to quantify the benefit of more advanced Machine Learning models.

In [ ]:
# ==========================================================
# Linear Regression
# ==========================================================

linear_model = LinearRegression()

linear_model.fit(

    X_train,

    y_train

)

linear_prediction = linear_model.predict(

    X_test

)

print("Linear Regression model trained.")

In [ ]:
linear_mae = mean_absolute_error(

    y_test,

    linear_prediction

)

linear_rmse = np.sqrt(

    mean_squared_error(

        y_test,

        linear_prediction

    )

)

linear_r2 = r2_score(

    y_test,

    linear_prediction

)

print(f"MAE  : {linear_mae:.2f}")

print(f"RMSE : {linear_rmse:.2f}")

print(f"R²   : {linear_r2:.3f}")

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(

    y_test,

    linear_prediction,

    alpha=0.5,

    s=10

)

mn = min(

    y_test.min(),

    linear_prediction.min()

)

mx = max(

    y_test.max(),

    linear_prediction.max()

)

plt.plot(

    [mn,mx],

    [mn,mx],

    "r--",

    linewidth=2

)

plt.xlabel("Observed TEC")

plt.ylabel("Predicted TEC")

plt.title("Linear Regression")

plt.grid(True)

plt.show()

## Discussion

Linear Regression captures the broad relationship between the predictors and TEC, but many observations deviate from the one-to-one line.

This is expected because the equatorial ionosphere is governed by complex nonlinear processes.

Can we improve the prediction by using a nonlinear Machine Learning model?

# Random Forest Regression

Random Forest is an ensemble learning algorithm.

Instead of building one decision tree,

it builds hundreds of decision trees and combines their predictions.

Advantages

- Handles nonlinear relationships
- Robust to noisy observations
- Requires little data preprocessing
- Often performs very well for environmental datasets

In [ ]:
# ==========================================================
# Random Forest
# ==========================================================

rf_model = RandomForestRegressor(

    n_estimators=300,

    max_depth=20,

    random_state=42,

    n_jobs=-1

)

rf_model.fit(

    X_train,

    y_train

)

rf_prediction = rf_model.predict(

    X_test

)

print("Random Forest model trained.")

In [ ]:
rf_mae = mean_absolute_error(

    y_test,

    rf_prediction

)

rf_rmse = np.sqrt(

    mean_squared_error(

        y_test,

        rf_prediction

    )

)

rf_r2 = r2_score(

    y_test,

    rf_prediction

)

print(f"MAE  : {rf_mae:.2f}")

print(f"RMSE : {rf_rmse:.2f}")

print(f"R²   : {rf_r2:.3f}")

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(

    y_test,

    rf_prediction,

    alpha=0.5,

    s=10

)

mn = min(

    y_test.min(),

    rf_prediction.min()

)

mx = max(

    y_test.max(),

    rf_prediction.max()

)

plt.plot(

    [mn,mx],

    [mn,mx],

    "r--",

    linewidth=2

)

plt.xlabel("Observed TEC")

plt.ylabel("Predicted TEC")

plt.title("Random Forest")

plt.grid(True)

plt.show()

## Scientific Interpretation

Compared with Linear Regression,

Random Forest typically produces predictions that lie much closer to the one-to-one line.

This improvement demonstrates that the relationship between space weather drivers and TEC is nonlinear.

Rather than assuming a simple linear relationship,

Random Forest learns complex interactions among the predictors.

## Transition

We have shown that a nonlinear ensemble method outperforms a simple linear model.

The next question is:

> **Can we do even better?**

In the next section we compare **Random Forest**, **Gradient Boosting**, and **XGBoost** under identical conditions to determine which model provides the most accurate TEC predictions.

# Part 4 — Comparing Machine Learning Models

So far we have trained

- Linear Regression
- Random Forest

A common question in Artificial Intelligence is

> Which algorithm should we use?

There is no universal answer.

Instead,

scientists compare several models under identical conditions.

Today we compare

- Linear Regression
- Random Forest
- Gradient Boosting
- XGBoost (optional)

using exactly the same training and testing datasets.

In [ ]:
# ==========================================================
# Gradient Boosting
# ==========================================================

gb_model = GradientBoostingRegressor(

    random_state=42

)

gb_model.fit(

    X_train,

    y_train

)

gb_prediction = gb_model.predict(

    X_test

)

print("Gradient Boosting model trained.")

In [ ]:
gb_mae = mean_absolute_error(

    y_test,

    gb_prediction

)

gb_rmse = np.sqrt(

    mean_squared_error(

        y_test,

        gb_prediction

    )

)

gb_r2 = r2_score(

    y_test,

    gb_prediction

)

print(f"MAE  : {gb_mae:.2f}")

print(f"RMSE : {gb_rmse:.2f}")

print(f"R²   : {gb_r2:.3f}")

## Optional: XGBoost

XGBoost (Extreme Gradient Boosting) is one of the most successful machine learning algorithms for structured data.

It improves upon Gradient Boosting through

- regularization,
- efficient optimization,
- handling of missing values,
- and parallel computation.

If the `xgboost` package is unavailable, you may skip this section.

In [ ]:
# Optional

try:

    from xgboost import XGBRegressor

    xgb_model = XGBRegressor(

        objective="reg:squarederror",

        n_estimators=300,

        learning_rate=0.05,

        max_depth=6,

        random_state=42

    )

    xgb_model.fit(

        X_train,

        y_train

    )

    xgb_prediction = xgb_model.predict(

        X_test

    )

    xgb_available = True

except ImportError:

    print("XGBoost is not installed.")

    xgb_available = False

In [ ]:
if xgb_available:

    xgb_mae = mean_absolute_error(

        y_test,

        xgb_prediction

    )

    xgb_rmse = np.sqrt(

        mean_squared_error(

            y_test,

            xgb_prediction

        )

    )

    xgb_r2 = r2_score(

        y_test,

        xgb_prediction

    )

    print(f"MAE  : {xgb_mae:.2f}")

    print(f"RMSE : {xgb_rmse:.2f}")

    print(f"R²   : {xgb_r2:.3f}")

# Model Comparison

Rather than examining each model individually,

let us compare all models in one table.

In [ ]:
results = pd.DataFrame({

    "Model":[

        "Linear Regression",

        "Random Forest",

        "Gradient Boosting"

    ],

    "MAE":[

        linear_mae,

        rf_mae,

        gb_mae

    ],

    "RMSE":[

        linear_rmse,

        rf_rmse,

        gb_rmse

    ],

    "R²":[

        linear_r2,

        rf_r2,

        gb_r2

    ]

})

if xgb_available:

    results.loc[len(results)] = [

        "XGBoost",

        xgb_mae,

        xgb_rmse,

        xgb_r2

    ]

results

In [ ]:
results.sort_values(

    "R²",

    ascending=False

)

In [ ]:
plt.figure(figsize=(8,4))

plt.bar(

    results["Model"],

    results["R²"]

)

plt.ylabel("R²")

plt.title("Model Comparison")

plt.xticks(rotation=15)

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8,4))

plt.bar(

    results["Model"],

    results["MAE"]

)

plt.ylabel("MAE")

plt.title("Mean Absolute Error")

plt.xticks(rotation=15)

plt.grid(True)

plt.show()

## Discussion

Notice that

- Linear Regression provides a useful baseline.
- Random Forest usually performs much better.
- Gradient Boosting often provides a further improvement.
- XGBoost frequently achieves the highest accuracy.

Tree-based ensemble methods generally outperform linear models because they naturally learn nonlinear relationships among the predictors.

# Step 5 — Explainable Artificial Intelligence

One criticism of Artificial Intelligence is that it behaves like a

**black box.**

Scientists naturally ask

> Why did the model predict this value?

To answer that question,

we use Explainable Artificial Intelligence (XAI).

In [ ]:
importance = pd.Series(

    rf_model.feature_importances_,

    index=features

).sort_values()

importance

In [ ]:
plt.figure(figsize=(8,5))

importance.plot.barh()

plt.xlabel("Relative Importance")

plt.title("Random Forest Feature Importance")

plt.show()

## Interpreting Feature Importance

The plot ranks the predictors according to their contribution to the Random Forest model.

Ask yourself:

- Does the ranking agree with ionospheric physics?
- Why might **Hour** and **DayOfYear** be highly important?
- Why is **F10.7** influential?
- Why does **Dst** matter during disturbed periods?

Feature importance provides a bridge between machine learning and physical interpretation.

# Beyond Feature Importance

Feature importance tells us **which variables matter**.

It does not tell us **how** they influence individual predictions.

For that purpose we use

**SHAP (SHapley Additive exPlanations).**

In [ ]:
# Install once if necessary
!pip install shap

In [ ]:
import shap

# To speed up the SHAP calculation, we will use a sample of the test data
X_test_sample = X_test.sample(n=100, random_state=42)

explainer = shap.TreeExplainer(rf_model)

shap_values = explainer.shap_values(X_test_sample)

In [ ]:
shap.summary_plot(

    shap_values,

    X_test_sample

)

## Scientific Interpretation of SHAP

The SHAP summary plot provides two types of information:

### 1. Feature Importance

Variables are ranked according to their overall contribution.

### 2. Direction of Influence

Each point represents one observation.

- Red points correspond to high feature values.
- Blue points correspond to low feature values.

The horizontal position indicates whether a feature increases or decreases the predicted TEC.

This makes SHAP much more informative than simple feature importance alone and helps relate the machine learning model back to the underlying ionospheric physics.

# Step 6 — Operational Space Weather Prediction

So far we have

- explored the data,
- trained Machine Learning models,
- compared algorithms,
- interpreted the predictions.

The final question is

> **Can we use the model operationally?**

Imagine that today's solar and geomagnetic observations are available.

Can the model estimate the expected Total Electron Content (TEC)?

This is the basic principle behind modern AI-assisted space weather forecasting systems.

In [ ]:
# ==========================================================
# Example Operational Space Weather Inputs
# ==========================================================

today = pd.DataFrame({

    "Hour":[14],

    "DayOfYear":[220],

    "F107":[180],

    "Dst":[-40],

    "Kp":[4],

    "Bz":[-6],

    "Vsw":[550],

    "Density":[8]

})

today

In [ ]:
forecast = rf_model.predict(today)

print(f"Predicted TEC = {forecast[0]:.2f} TECU")

## Interpretation

The model has produced a TEC estimate using only

- solar activity,
- geomagnetic activity,
- solar wind,
- time.

No TEC measurement was supplied.

This demonstrates how Artificial Intelligence can support operational forecasting.

In practice,

new solar wind observations become available continuously,

allowing predictions to be updated every hour.

In [ ]:
plt.figure(figsize=(8, 6))

plt.hist(data['TEC'], bins=50, alpha=0.7, color='skyblue', label='Historical TEC Distribution', density=True)
plt.axvline(forecast[0], color='red', linestyle='--', linewidth=2, label=f'Forecasted TEC = {forecast[0]:.2f} TECU')

plt.xlabel('TEC (TECU)')
plt.ylabel('Density')
plt.title('Operational TEC Forecast in Context of Historical Data')
plt.legend()
plt.grid(True)
plt.show()

# What Have We Learned?

Today we completed an end-to-end Artificial Intelligence workflow.

We

✓ generated a realistic space weather dataset

✓ explored the observations

✓ trained multiple Machine Learning models

✓ compared their performance

✓ interpreted the results

✓ generated an operational TEC forecast

This is workflow can used in many environmental and geophysical prediction problems.

## Suggestions for Further Analysis

1. **Feature Ablation Study**
   - Remove key predictors (e.g., **F10.7**, **Dst**, or **Hour**) and retrain the model to evaluate their impact on prediction accuracy. This demonstrates the physical significance of each variable and reinforces the connection between feature importance and ionospheric physics.

2. **Cross-Validation**
   - Replace a single train/test split with **k-fold cross-validation** (e.g., 5-fold) to obtain a more robust estimate of model performance and assess model generalization across multiple data partitions.

3. **Time-Based Train/Test Split**
   - Use a chronological train/test split instead of a random split. Since space weather is inherently a time series, training on earlier observations and testing on later observations better reflects operational forecasting, prevents information leakage, and provides a more realistic evaluation of model performance.

# Limitations

Although our models perform well,

they have important limitations.

## Data

Our synthetic dataset is designed for teaching.

Real observations contain

- missing values,
- instrument errors,
- calibration issues,
- changing climatology.

---

## Physics

The models are statistical.

They do not explicitly solve

- Maxwell's equations,
- continuity equations,
- momentum equations,
- plasma transport.

Therefore,

Machine Learning should complement rather than replace physical models.

# Future Directions

The next generation of AI-assisted space weather prediction will combine

## Better Data

- GNSS networks
- Swarm satellites
- Ionosondes
- COSMIC radio occultation

---

## Better Algorithms

- XGBoost
- CatBoost
- LightGBM

---

## Deep Learning

- LSTM
- GRU
- Transformers

These models can learn temporal dependencies that tree-based models do not explicitly capture.

---

## Explainable AI

- SHAP
- LIME
- Counterfactual explanations

These methods help scientists understand why a prediction was made.

---

## Physics-Informed AI

The future lies in combining

Physics

+

Observations

+

Machine Learning

to produce accurate and physically interpretable operational forecasts.

# Key Takeaways

The equatorial ionosphere is a highly nonlinear system.

Artificial Intelligence provides powerful tools for analysing and predicting its behaviour.

However,

Artificial Intelligence is **most effective when combined with physical understanding**.

The workflow you learned today can be applied to many space weather problems, including

- Total Electron Content (TEC)
- foF2 prediction
- hmF2 prediction
- Spread-F detection
- Plasma bubble forecasting
- Scintillation prediction
- GNSS positioning error estimation

The methodology remains the same:

**Understand the science → Understand the data → Build the model → Evaluate the model → Interpret the results → Apply the model operationally.**

# Exercises

---

Try the following after this workshop.

### Exercise 1

Increase

```python
n_estimators = 1000
```

Does Random Forest improve?

---

### Exercise 2

Replace Random Forest with

```python
ExtraTreesRegressor()
```

Compare the results.

---

### Exercise 3

Remove one feature

```python
F107
```

Does the prediction deteriorate?

Why?

---

### Exercise 4

Predict

instead of TEC

- foF2
- hmF2
- S4
- Plasma Bubble Occurrence

What changes?

---

### Exercise 5

Train an LSTM using sequences of the previous 24 hours.

Does temporal information improve the forecast?

---

# Recommended Reading

---

## Space Physics

- Schunk, R. W., & Nagy, A. F. (2009). *Ionospheres: Physics, plasma physics, and chemistry* (2nd ed.). Cambridge University Press.

- Kelley, M. C. (2009). *The Earth's ionosphere: Plasma physics and electrodynamics* (2nd ed.). Academic Press.

## Machine Learning

- Géron, A. (2019). *Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow: Concepts, tools, and techniques to build intelligent systems* (2nd ed.). O'Reilly Media.

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An introduction to statistical learning: With applications in R* (2nd ed.). Springer.

---


# Final Message

Artificial Intelligence will not replace space physicists.

**It will enable space physicists to solve problems that were previously intractable.**

---

# Thank you!

# Questions?

---